In [1]:
# Cellule 1 — Config & Imports

In [2]:
# ══════════════════════════════════════════════════════════════
#  NOTEBOOK 03 — ENTRAÎNEMENT, ÉVALUATION, PERFORMANCES
#  Un seul paramètre à changer entre les bases
# ══════════════════════════════════════════════════════════════

# ── Seul paramètre à modifier ─────────────────────────────────
BASE_NAME     = "STE_NGDM"
PARETO_SEUIL  = 0.80    # top articles représentant 80% des sorties
TEST_RATIO    = 0.20    # 20% pour le test
CONTAMINATION = 0.05    # Isolation Forest
RF_ESTIMATORS = 100     # Random Forest

# ── Chemins ───────────────────────────────────────────────────
DATA_DIR  = f"../../output/{BASE_NAME}/data_clean/"
MODEL_DIR = f"../../models/{BASE_NAME}/"
PERF_DIR  = f"../../output/{BASE_NAME}/performance/"
PLOT_DIR  = f"../../output/{BASE_NAME}/plots/"

import os
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PERF_DIR,  exist_ok=True)
os.makedirs(PLOT_DIR,  exist_ok=True)

# ── Manipulation données ──────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── Prophet ───────────────────────────────────────────────────
from prophet import Prophet
import holidays

# ── Random Forest ─────────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── Isolation Forest ──────────────────────────────────────────
from sklearn.ensemble import IsolationForest

# ── K-Means ───────────────────────────────────────────────────
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ── Sauvegarde modèles ────────────────────────────────────────
import pickle
import warnings
warnings.filterwarnings('ignore')

# ── Style graphes ─────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi']     = 120

print("✅ Imports OK")
print(f"📦 Base ciblée : {BASE_NAME}")
print(f"📁 Data        : {DATA_DIR}")
print(f"📁 Models      : {MODEL_DIR}")

d:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


✅ Imports OK
📦 Base ciblée : STE_NGDM
📁 Data        : ../../output/STE_NGDM/data_clean/
📁 Models      : ../../models/STE_NGDM/


In [ ]:
# Cellule 2 — Chargement des datasets

In [ ]:
df_prophet = pd.read_csv(f"{DATA_DIR}df_prophet.csv",  parse_dates=['ds'])
df_rf      = pd.read_csv(f"{DATA_DIR}df_rf.csv",       parse_dates=['DateJour'])
df_iforest = pd.read_csv(f"{DATA_DIR}df_iforest.csv",  parse_dates=['DateJour'])
df_kmeans  = pd.read_csv(f"{DATA_DIR}df_kmeans.csv",   index_col=0)

print("✅ Datasets chargés")
print(f"   df_prophet  : {len(df_prophet):,} lignes | {df_prophet['AR_Ref'].nunique()} articles")
print(f"   df_rf       : {len(df_rf):,} lignes | {df_rf['AR_Ref'].nunique()} articles")
print(f"   df_iforest  : {len(df_iforest):,} lignes | {df_iforest['AR_Ref'].nunique()} articles")
print(f"   df_kmeans   : {df_kmeans.shape[0]} articles × {df_kmeans.shape[1]} mois")

In [ ]:
# Cellule 4 — Jours fériés marocains

In [ ]:
# ── Jours fériés Maroc ────────────────────────────────────────
# Calculés dynamiquement sur toute la période des données

annees = df_prophet['ds'].dt.year.unique().tolist()

ma_holidays = holidays.Morocco(years=annees)
df_holidays = pd.DataFrame({
    'ds'      : pd.to_datetime(list(ma_holidays.keys())),
    'holiday' : list(ma_holidays.values())
})
df_holidays = df_holidays.sort_values('ds').reset_index(drop=True)

print(f"✅ Jours fériés marocains chargés")
print(f"   Années couvertes : {min(annees)} → {max(annees)}")
print(f"   Nb jours fériés  : {len(df_holidays)}")

In [ ]:
# Cellule 5 — Entraînement Prophet

In [ ]:
# ── Résultats Prophet ─────────────────────────────────────────
resultats_prophet = []

for ar_ref in top_articles:

    # ── Données de l'article ──────────────────────────────────
    df_art = df_prophet[df_prophet['AR_Ref'] == ar_ref][['ds', 'y']].copy()
    df_art = df_art.sort_values('ds').reset_index(drop=True)

    # ── Split train/test dynamique ────────────────────────────
    cutoff    = df_art['ds'].max() - pd.Timedelta(days=int(len(df_art) * TEST_RATIO))
    df_train  = df_art[df_art['ds'] <= cutoff]
    df_test   = df_art[df_art['ds'] >  cutoff]

    # ── Configurer Prophet ────────────────────────────────────
    model = Prophet(
        weekly_seasonality       = True,
        yearly_seasonality       = True,
        changepoint_prior_scale  = 0.05,
        holidays                 = df_holidays
    )
    model.fit(df_train)

    # ── Prédire sur la période de test ────────────────────────
    future   = model.make_future_dataframe(periods=len(df_test))
    forecast = model.predict(future)
    forecast_test = forecast[forecast['ds'].isin(df_test['ds'])]

    # ── Métriques ─────────────────────────────────────────────
    y_true = df_test['y'].values
    y_pred = forecast_test['yhat'].values[:len(y_true)]
    y_pred = np.maximum(y_pred, 0)  # pas de prédiction négative

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

    resultats_prophet.append({
        'AR_Ref' : ar_ref,
        'MAE'    : round(mae,  2),
        'RMSE'   : round(rmse, 2),
        'MAPE'   : round(mape, 2),
        'nb_train': len(df_train),
        'nb_test' : len(df_test)
    })

    # ── Sauvegarder le modèle ─────────────────────────────────
    model_path = f"{MODEL_DIR}prophet_{ar_ref}.pkl"
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)

    print(f"   {ar_ref:15s} | MAE={mae:.1f} | RMSE={rmse:.1f} | MAPE={mape:.1f}%")

df_perf_prophet = pd.DataFrame(resultats_prophet)
print(f"\n✅ Prophet : {len(top_articles)} modèles entraînés")
print(f"   MAE moyen  : {df_perf_prophet['MAE'].mean():.1f}")
print(f"   RMSE moyen : {df_perf_prophet['RMSE'].mean():.1f}")
print(f"   MAPE moyen : {df_perf_prophet['MAPE'].mean():.1f}%")

In [ ]:
# Cellule 6 — Graphes Prophet (top 3 articles)

In [ ]:
# ── Tracer les 3 meilleurs articles ───────────────────────────
top3 = df_perf_prophet.nsmallest(3, 'MAPE')['AR_Ref'].tolist()

for ar_ref in top3:
    df_art   = df_prophet[df_prophet['AR_Ref'] == ar_ref][['ds','y']].sort_values('ds')
    cutoff   = df_art['ds'].max() - pd.Timedelta(days=int(len(df_art) * TEST_RATIO))
    df_train = df_art[df_art['ds'] <= cutoff]
    df_test  = df_art[df_art['ds'] >  cutoff]

    with open(f"{MODEL_DIR}prophet_{ar_ref}.pkl", 'rb') as f:
        model = pickle.load(f)

    future   = model.make_future_dataframe(periods=len(df_test) + 30)
    forecast = model.predict(future)

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df_train['ds'], df_train['y'], color='steelblue',  label='Train')
    ax.plot(df_test['ds'],  df_test['y'],  color='green',      label='Réel (test)')
    ax.plot(forecast['ds'], forecast['yhat'], color='orange', linestyle='--', label='Prédit')
    ax.fill_between(forecast['ds'],
                    forecast['yhat_lower'],
                    forecast['yhat_upper'],
                    alpha=0.2, color='orange', label='Intervalle confiance')
    ax.set_title(f'Prophet — {ar_ref} | MAPE={df_perf_prophet[df_perf_prophet["AR_Ref"]==ar_ref]["MAPE"].values[0]:.1f}%')
    ax.set_xlabel('Date')
    ax.set_ylabel('TotalSortie')
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}prophet_{ar_ref}.png", dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# Cellule 7 — Random Forest

In [ ]:
# ── Features disponibles dynamiquement ───────────────────────
features_voulues = ['sortie_lag_1', 'sortie_lag_7', 'sortie_lag_30',
                    'rolling_mean_7', 'rolling_mean_30',
                    'StockFinal', 'jour_semaine', 'mois', 'trimestre']

features_rf = [col for col in features_voulues if col in df_rf.columns]
target_rf   = 'jours_avant_rupture'

print(f"Features utilisées : {features_rf}")

# ── Supprimer lignes sans target ──────────────────────────────
df_rf_clean = df_rf.dropna(subset=[target_rf] + features_rf).copy()
df_rf_clean = df_rf_clean.sort_values('DateJour').reset_index(drop=True)

print(f"Lignes après nettoyage : {len(df_rf_clean):,}")

# ── Split temporel ────────────────────────────────────────────
cutoff_idx = int(len(df_rf_clean) * (1 - TEST_RATIO))
train_rf   = df_rf_clean.iloc[:cutoff_idx]
test_rf    = df_rf_clean.iloc[cutoff_idx:]

X_train = train_rf[features_rf]
y_train = train_rf[target_rf]
X_test  = test_rf[features_rf]
y_test  = test_rf[target_rf]

# ── Normalisation ─────────────────────────────────────────────
scaler   = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ── Entraînement ──────────────────────────────────────────────
rf_model = RandomForestRegressor(
    n_estimators = RF_ESTIMATORS,
    random_state = 42,
    n_jobs       = -1
)
rf_model.fit(X_train_s, y_train)

# ── Métriques ─────────────────────────────────────────────────
y_pred_rf = rf_model.predict(X_test_s)
mae_rf    = mean_absolute_error(y_test, y_pred_rf)
rmse_rf   = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f"\n✅ Random Forest entraîné")
print(f"   Train : {len(train_rf):,} lignes")
print(f"   Test  : {len(test_rf):,} lignes")
print(f"   MAE   : {mae_rf:.1f} jours")
print(f"   RMSE  : {rmse_rf:.1f} jours")

# ── Sauvegarder ───────────────────────────────────────────────
with open(f"{MODEL_DIR}rf_rupture.pkl", 'wb') as f:
    pickle.dump(rf_model, f)
with open(f"{MODEL_DIR}scaler_rf.pkl", 'wb') as f:
    pickle.dump(scaler, f)

print(f"✅ Modèles sauvegardés dans {MODEL_DIR}")

In [ ]:
# Cellule 8 — Graphe feature importance RF

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

importances = pd.Series(rf_model.feature_importances_, index=features_rf)
importances = importances.sort_values(ascending=True)

importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title(f'Feature Importance — Random Forest ({BASE_NAME})')
ax.set_xlabel('Importance')

plt.tight_layout()
plt.savefig(f"{PLOT_DIR}rf_feature_importance.png", dpi=120, bbox_inches='tight')
plt.show()
print("✅ Graphe sauvegardé : rf_feature_importance.png")

In [ ]:
# Cellule 10 — Graphe Isolation Forest

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Isolation Forest — Anomalies ({BASE_NAME})', fontsize=14)

# ── Scatter TotalSortie vs StockFinal coloré par anomalie ─────
colors = df_iforest['is_anomalie'].map({0: 'steelblue', 1: 'red'})
axes[0].scatter(df_iforest['TotalSortie'],
                df_iforest['StockFinal'],
                c=colors, alpha=0.3, s=5)
axes[0].set_title('TotalSortie vs StockFinal')
axes[0].set_xlabel('TotalSortie')
axes[0].set_ylabel('StockFinal')

# Légende manuelle
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(color='steelblue', label='Normal'),
    Patch(color='red',       label='Anomalie')
])

# ── Distribution anomaly_score ────────────────────────────────
axes[1].hist(df_iforest['anomaly_score'], bins=50,
             color='steelblue', edgecolor='white')
axes[1].axvline(x=0, color='red', linestyle='--', label='Seuil anomalie')
axes[1].set_title('Distribution Anomaly Score')
axes[1].set_xlabel('Score')
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{PLOT_DIR}iforest_anomalies.png", dpi=120, bbox_inches='tight')
plt.show()
print("✅ Graphe sauvegardé : iforest_anomalies.png")

In [ ]:
# Cellule 11 — K-Means : méthode Elbow

In [ ]:
# ── Normalisation ─────────────────────────────────────────────
scaler_km  = StandardScaler()
X_km       = scaler_km.fit_transform(df_kmeans.fillna(0))

# ── Elbow : tester k=2 à 10 ───────────────────────────────────
k_range    = range(2, min(11, len(df_kmeans)))
inertias   = []
silhouettes = []

for k in k_range:
    km   = KMeans(n_clusters=k, random_state=42, n_init=10)
    labs = km.fit_predict(X_km)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_km, labs))

# ── Graphe Elbow + Silhouette ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle(f'K-Means : Choix k optimal — {BASE_NAME}', fontsize=14)

axes[0].plot(list(k_range), inertias, marker='o', color='steelblue')
axes[0].set_title('Méthode Elbow (Inertie)')
axes[0].set_xlabel('Nombre de clusters k')
axes[0].set_ylabel('Inertie')

axes[1].plot(list(k_range), silhouettes, marker='o', color='coral')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Nombre de clusters k')
axes[1].set_ylabel('Score')

plt.tight_layout()
plt.savefig(f"{PLOT_DIR}kmeans_elbow.png", dpi=120, bbox_inches='tight')
plt.show()

# ── k optimal = silhouette max ────────────────────────────────
k_optimal = list(k_range)[silhouettes.index(max(silhouettes))]
print(f"✅ k optimal détecté automatiquement : {k_optimal}")
print(f"   Silhouette score : {max(silhouettes):.3f}")

In [ ]:
# Cellule 12 — K-Means : entraînement

In [ ]:
# ── Entraînement avec k optimal ───────────────────────────────
km_final = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
labels   = km_final.fit_predict(X_km)

df_kmeans_result         = df_kmeans.copy()
df_kmeans_result['cluster'] = labels

# ── Nommer les clusters automatiquement ───────────────────────
# Calculer sortie moyenne par cluster → trier → nommer
sortie_par_cluster = df_kmeans_result.groupby('cluster')\
    .apply(lambda x: x.drop(columns='cluster').sum(axis=1).mean())

cluster_rank = sortie_par_cluster.sort_values(ascending=False)

noms_disponibles = ['forte rotation', 'rotation moyenne',
                    'faible rotation', 'quasi-immobile']

# Adapter le nombre de noms au nombre de clusters
noms_clusters = {
    cluster: noms_disponibles[i] if i < len(noms_disponibles) else f'cluster_{i}'
    for i, cluster in enumerate(cluster_rank.index)
}

df_kmeans_result['segment'] = df_kmeans_result['cluster'].map(noms_clusters)

print(f"✅ K-Means entraîné — k={k_optimal}")
print("\nRépartition par segment :")
for nom, grp in df_kmeans_result.groupby('segment'):
    print(f"   {nom:20s} : {len(grp)} articles")

# ── Sauvegarder ───────────────────────────────────────────────
with open(f"{MODEL_DIR}kmeans_segments.pkl", 'wb') as f:
    pickle.dump({'model': km_final, 'scaler': scaler_km,
                 'noms_clusters': noms_clusters}, f)

print(f"\n✅ Modèle sauvegardé : kmeans_segments.pkl")

In [ ]:
# Cellule 13 — Graphes K-Means

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'K-Means Segmentation — {BASE_NAME}', fontsize=14)

# ── Nb articles par segment ───────────────────────────────────
seg_counts = df_kmeans_result['segment'].value_counts()
axes[0].bar(seg_counts.index, seg_counts.values, color='steelblue')
axes[0].set_title('Nb articles par segment')
axes[0].set_ylabel('Nb articles')
axes[0].tick_params(axis='x', rotation=30)

# ── Sortie moyenne par segment ────────────────────────────────
seg_means = df_kmeans_result.groupby('segment')\
    .apply(lambda x: x.drop(columns=['cluster','segment']).sum(axis=1).mean())
axes[1].bar(seg_means.index, seg_means.values, color='coral')
axes[1].set_title('Sortie mensuelle moyenne par segment')
axes[1].set_ylabel('TotalSortie moyen')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(f"{PLOT_DIR}kmeans_segments.png", dpi=120, bbox_inches='tight')
plt.show()
print("✅ Graphe sauvegardé : kmeans_segments.png")

In [ ]:
# Cellule 14 — Export performances

In [ ]:
# ── Prophet ───────────────────────────────────────────────────
df_perf_prophet.to_csv(f"{PERF_DIR}perf_prophet.csv", index=False)

# ── Random Forest ─────────────────────────────────────────────
df_perf_rf = pd.DataFrame([{
    'model' : 'RandomForest',
    'MAE'   : round(mae_rf,  2),
    'RMSE'  : round(rmse_rf, 2),
    'nb_train': len(train_rf),
    'nb_test' : len(test_rf)
}])
df_perf_rf.to_csv(f"{PERF_DIR}perf_rf.csv", index=False)

# ── Isolation Forest ──────────────────────────────────────────
df_perf_if = pd.DataFrame([{
    'model'        : 'IsolationForest',
    'contamination': CONTAMINATION,
    'nb_anomalies' : int(nb_anomalies),
    'pct_anomalies': round(nb_anomalies/len(df_iforest)*100, 2)
}])
df_perf_if.to_csv(f"{PERF_DIR}perf_iforest.csv", index=False)

# ── K-Means ───────────────────────────────────────────────────
df_perf_km = df_kmeans_result['segment'].value_counts()\
    .reset_index().rename(columns={'index':'segment', 'segment':'nb_articles'})
df_perf_km.to_csv(f"{PERF_DIR}perf_kmeans.csv", index=False)

print(f"✅ Performances exportées dans {PERF_DIR}")
print(f"   perf_prophet.csv")
print(f"   perf_rf.csv")
print(f"   perf_iforest.csv")
print(f"   perf_kmeans.csv")

In [ ]:
# Cellule 15 — Résumé final

In [ ]:
print("=" * 60)
print(f"  RÉSUMÉ ENTRAÎNEMENT — {BASE_NAME}")
print("=" * 60)
print(f"""
🔮 Prophet
   Articles entraînés  : {len(top_articles)}
   MAE moyen           : {df_perf_prophet['MAE'].mean():.1f}
   MAPE moyen          : {df_perf_prophet['MAPE'].mean():.1f}%

🌲 Random Forest
   MAE                 : {mae_rf:.1f} jours
   RMSE                : {rmse_rf:.1f} jours

🔍 Isolation Forest
   Anomalies détectées : {int(nb_anomalies)} ({nb_anomalies/len(df_iforest)*100:.1f}%)

🎯 K-Means
   k optimal           : {k_optimal}
   Segments            : {list(noms_clusters.values())}

📁 Modèles sauvegardés dans : {MODEL_DIR}
📁 Performances dans        : {PERF_DIR}
""")